
# 練習問題: teams / distribute / parallel / for の出力行数を予測する

## 目標

`teams`, `distribute`, `parallel`, `for` を入れ子にしたとき, それぞれの領域が「何回」実行されるかを正しく理解する.
この問題は**コードを書く問題ではなく, 出力行数を予測してから実測で確かめる読解・予測の問題**である.

## プログラムの構造

`teams_quiz.cpp` (または `teams_quiz.f90`) は完成済みで, 次の入れ子構造を持つ.

```
#pragma omp target teams            // T 個のチームを作る
  in teams を表示
  #pragma omp distribute            // i = 0 .. m-1 をチーム間で分配
  for i in 0..m-1
    in distribute を表示
    #pragma omp parallel num_threads(H)   // 各チーム内に H 本のスレッド
      in parallel を表示
    #pragma omp for                 // j = 0 .. n-1 をスレッド間で分配
    for j in 0..n-1
      in for を表示
```

- チーム数は `OMP_NUM_TEAMS` で指定する (これを $T$ とする).
- 1チームあたりのスレッド数は `OMP_NUM_THREADS` で指定する (これを $H$ とする). `num_threads()` 経由でプログラムに渡している.
  - `OMP_NUM_THREADS` は **1 または 32 の倍数** でなければならない (GPUのスレッドは32本単位(ワープ)で動くため). それ以外を指定するとプログラムがエラーで止まる.
- コマンドライン引数 `m`, `n` がループ回数になる.

## 課題

`OMP_NUM_TEAMS=T`, `OMP_NUM_THREADS=H`, 引数 `m`, `n` で実行したとき, 次の各メッセージが何行表示されるかを **$T$, $H$, $m$, $n$ の式で予測** せよ.

- `in teams:` の行数
- `in distribute:` の行数
- `in parallel:` の行数
- `in for:` の行数
- 合計行数

ヒント: `in teams` はチームごとに1回. `distribute` と `for` はループの繰り返しを分配する (繰り返しの総数だけ表示される) のに対し, `parallel` は各チーム内でスレッドを作る (チーム内のスレッドの数だけ表示される) ことに注意せよ.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu teams_quiz.cpp -o teams_quiz.exe

# Fortran
nvfortran -mp=gpu teams_quiz.f90 -o teams_quiz.exe
```

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入して実行する.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

OMP_NUM_TEAMS=2 OMP_NUM_THREADS=32 ./teams_quiz.exe 5 6
```

## 予測の確認

`wc -l` で実際の行数を数え, 予測と一致するか確かめよ. メッセージごとに `grep` で絞り込むと数えやすい.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

OMP_NUM_TEAMS=2 OMP_NUM_THREADS=32 ./teams_quiz.exe 5 6 | wc -l
OMP_NUM_TEAMS=2 OMP_NUM_THREADS=32 ./teams_quiz.exe 5 6 | grep "in teams"      | wc -l
OMP_NUM_TEAMS=2 OMP_NUM_THREADS=32 ./teams_quiz.exe 5 6 | grep "in distribute" | wc -l
OMP_NUM_TEAMS=2 OMP_NUM_THREADS=32 ./teams_quiz.exe 5 6 | grep "in parallel"   | wc -l
OMP_NUM_TEAMS=2 OMP_NUM_THREADS=32 ./teams_quiz.exe 5 6 | grep "in for"        | wc -l
```

$T$, $H$, $m$, $n$ をいろいろ変えて実行し, 予測した式が常に成り立つことを確認せよ.
表示される行の `team` 番号や `thread` 番号にも注目すると, どの繰り返しがどのチーム・スレッドに割り当てられたかが分かる.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ teams_quiz.cpp
#include <cstdio>
#include <cstdlib>
#include <omp.h>

int main(int argc, char ** argv) {
  char * nthreads_ = getenv("OMP_NUM_THREADS");
  int    nthreads  = (nthreads_ ? atoi(nthreads_) : 1);
  if (nthreads != 1 && nthreads % 32) {
    fprintf(stderr, "OMP_NUM_THREADS (%d) must be 1 or a multiple of 32\n", nthreads);
    exit(1);
  }
  int m = (argc > 1 ? atoi(argv[1]) : 5);
  int n = (argc > 2 ? atoi(argv[2]) : 6);
#pragma omp target teams
  {
    printf("in teams: team %03d/%03d\n",
           omp_get_team_num(), omp_get_num_teams());
#pragma omp distribute
    for (int i = 0; i < m; i++) {
      printf("in distribute: i=%03d team %03d/%03d\n",
             i, omp_get_team_num(), omp_get_num_teams());
#pragma omp parallel num_threads(nthreads)
      printf("in parallel: i=%03d team %03d/%03d thread %03d/%03d\n",
             i, omp_get_team_num(), omp_get_num_teams(),
             omp_get_thread_num(), omp_get_num_threads());
#pragma omp for
      for (int j = 0; j < n; j++) {
        printf("in for: i=%03d j=%03d team %03d/%03d thread %03d/%03d\n",
               i, j, omp_get_team_num(), omp_get_num_teams(),
               omp_get_thread_num(), omp_get_num_threads());
      }
    }
  }
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu teams_quiz.cpp -o teams_quiz_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./teams_quiz_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ teams_quiz.f90
program teams_quiz
  use omp_lib
  implicit none
  character(len=32) :: arg
  integer :: nthreads, m, n, i, j, stat

  nthreads = 1
  call get_environment_variable("OMP_NUM_THREADS", arg, status=stat)
  if (stat == 0) read (arg, *) nthreads
  if (nthreads /= 1 .and. mod(nthreads, 32) /= 0) then
     write (0, "(a,i0,a)") "OMP_NUM_THREADS (", nthreads, &
          ") must be 1 or a multiple of 32"
     stop 1
  end if

  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  else
     m = 5
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) n
  else
     n = 6
  end if

  !$omp target teams
  print "(a,i3.3,a,i3.3)", "in teams: team ", &
       omp_get_team_num(), "/", omp_get_num_teams()
  !$omp distribute
  do i = 0, m - 1
     print "(a,i3.3,a,i3.3,a,i3.3)", "in distribute: i=", i, &
          " team ", omp_get_team_num(), "/", omp_get_num_teams()
     !$omp parallel num_threads(nthreads)
     print "(a,i3.3,a,i3.3,a,i3.3,a,i3.3,a,i3.3)", "in parallel: i=", i, &
          " team ", omp_get_team_num(), "/", omp_get_num_teams(), &
          " thread ", omp_get_thread_num(), "/", omp_get_num_threads()
     !$omp end parallel
     !$omp do
     do j = 0, n - 1
        print "(a,i3.3,a,i3.3,a,i3.3,a,i3.3,a,i3.3,a,i3.3)", "in for: i=", i, &
             " j=", j, " team ", omp_get_team_num(), "/", omp_get_num_teams(), &
             " thread ", omp_get_thread_num(), "/", omp_get_num_threads()
     end do
     !$omp end do
  end do
  !$omp end distribute
  !$omp end target teams
end program teams_quiz

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu teams_quiz.f90 -o teams_quiz_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./teams_quiz_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:teams_quiz.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:teams_quiz.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:teams_quiz.cpp}